# Conditional Selection and Filtering

---

### Table of Contents
1. Introduction and Setup
2. The Logic of Boolean Indexing
3. Filtering with Multiple Conditions
4. Common Filtering Methods
5. The `.query()` Method

---

## 1. Introduction and Setup
- Filtering—selecting rows that meet certain criteria—is one of the most common tasks in data analysis.
- Pandas uses a powerful technique called "boolean indexing" or "masking" for this.

In [1]:
import os
import pandas as pd

# --- Load the sample dataset ---
DATA_FOLDER = "pandas_data"
file_path = os.path.join(DATA_FOLDER, "sample_sales_data.csv")

try:
    # We parse the 'OrderDate' column as dates right away.
    # For this lesson, we'll use the default integer index.
    df = pd.read_csv(file_path, parse_dates=["OrderDate"])
    print("--- Sample Sales DataFrame ---")
    print(df)
except FileNotFoundError:
    print(f"Error: The data file was not found at '{file_path}'")
    print("Please run '04_reading_and_writing_data.py' first to create it.")
    df = pd.DataFrame()  # Create an empty df to avoid further errors

--- Sample Sales DataFrame ---
   OrderID   Product     Category   Price  Quantity  OrderDate
0      101    Laptop  Electronics  1200.0         1 2025-01-15
1      102     Mouse  Electronics    25.5         2 2025-01-15
2      103  Keyboard  Electronics    75.0         1 2025-01-16
3      104   Monitor  Electronics   300.0         2 2025-01-17
4      105     Mouse  Accessories    27.0         3 2025-01-18
5      106    Webcam  Accessories    50.0         1 2025-01-18



---

## 2. The Logic of Boolean Indexing
- This is a two-step process:
  1. Create a logical condition on a Series (a column). This produces a new
     Series of `True`/`False` values, called a "boolean mask".
  2. Use this mask to index the DataFrame, which returns only the rows where the mask is `True`.

In [2]:
# --- Step 1: Create the boolean mask ---

# Let's find all rows where the Category is 'Electronics'.
mask = df["Category"] == "Electronics"
print("The boolean mask for `df['Category'] == 'Electronics'`:\n", mask)
print(f"Type of the mask: {type(mask)}")

The boolean mask for `df['Category'] == 'Electronics'`:
 0     True
1     True
2     True
3     True
4    False
5    False
Name: Category, dtype: bool
Type of the mask: <class 'pandas.core.series.Series'>


In [3]:
# --- Step 2: Apply the mask to the DataFrame ---

# This is typically done in a single, idiomatic line.
electronics_df = df[df["Category"] == "Electronics"]
print("\nDataFrame filtered for 'Electronics':\n", electronics_df)


DataFrame filtered for 'Electronics':
    OrderID   Product     Category   Price  Quantity  OrderDate
0      101    Laptop  Electronics  1200.0         1 2025-01-15
1      102     Mouse  Electronics    25.5         2 2025-01-15
2      103  Keyboard  Electronics    75.0         1 2025-01-16
3      104   Monitor  Electronics   300.0         2 2025-01-17



---

## 3. Filtering with Multiple Conditions
- To combine conditions, you MUST use:
  - `&` for `and`
  - `|` for `or`
  - `~` for `not`
- Each individual condition MUST be wrapped in parentheses `()`.

In [4]:
# --- Example with `&` (AND) ---

# Find all sales in the 'Electronics' category with a price over $100.
high_value_electronics = df[(df["Category"] == "Electronics") & (df["Price"] > 100)]
print(
    "High-value electronics (Category == 'Electronics' AND Price > 100):\n",
    high_value_electronics,
)

High-value electronics (Category == 'Electronics' AND Price > 100):
    OrderID  Product     Category   Price  Quantity  OrderDate
0      101   Laptop  Electronics  1200.0         1 2025-01-15
3      104  Monitor  Electronics   300.0         2 2025-01-17


In [5]:
# --- Example with `|` (OR) ---

# Find all sales that are either a 'Laptop' OR have a quantity of 3 or more.
laptops_or_high_qty = df[(df["Product"] == "Laptop") | (df["Quantity"] >= 3)]
print(
    "\nLaptops OR high quantity sales (Product == 'Laptop' OR Quantity >= 3):\n",
    laptops_or_high_qty,
)


Laptops OR high quantity sales (Product == 'Laptop' OR Quantity >= 3):
    OrderID Product     Category   Price  Quantity  OrderDate
0      101  Laptop  Electronics  1200.0         1 2025-01-15
4      105   Mouse  Accessories    27.0         3 2025-01-18



---

## 4. Common Filtering Methods
- Pandas provides convenient methods to simplify common filtering tasks.

In [6]:
# --- `.isin()` for checking against a list of values ---

# Find all sales for 'Laptop' or 'Webcam'
products_of_interest = ["Laptop", "Webcam"]
isin_filter_df = df[df["Product"].isin(products_of_interest)]
print("Using `.isin(['Laptop', 'Webcam'])`:\n", isin_filter_df)

Using `.isin(['Laptop', 'Webcam'])`:
    OrderID Product     Category   Price  Quantity  OrderDate
0      101  Laptop  Electronics  1200.0         1 2025-01-15
5      106  Webcam  Accessories    50.0         1 2025-01-18


In [7]:
# --- `.between()` for checking a numerical range (inclusive) ---

# Find all sales where the price was between $50 and $300.
between_filter_df = df[df["Price"].between(50, 300)]
print("\nUsing `.between(50, 300)` on Price:\n", between_filter_df)


Using `.between(50, 300)` on Price:
    OrderID   Product     Category  Price  Quantity  OrderDate
2      103  Keyboard  Electronics   75.0         1 2025-01-16
3      104   Monitor  Electronics  300.0         2 2025-01-17
5      106    Webcam  Accessories   50.0         1 2025-01-18


In [8]:
# --- String methods with the `.str` accessor ---

# The `.str` accessor lets you apply string methods to an entire Series.
# Find all products that contain the letter 'o'.
str_contains_df = df[df["Product"].str.contains("o", case=False)]  # case-insensitive
print("\nUsing `.str.contains('o')` on Product:\n", str_contains_df)


Using `.str.contains('o')` on Product:
    OrderID   Product     Category   Price  Quantity  OrderDate
0      101    Laptop  Electronics  1200.0         1 2025-01-15
1      102     Mouse  Electronics    25.5         2 2025-01-15
2      103  Keyboard  Electronics    75.0         1 2025-01-16
3      104   Monitor  Electronics   300.0         2 2025-01-17
4      105     Mouse  Accessories    27.0         3 2025-01-18



---

## 5. The `.query()` Method
- An alternative way to filter using a string expression.
- Can be more readable for complex conditions.

In [9]:
# This is equivalent to the `&` example above.
query_result = df.query('Category == "Electronics" and Price > 100')
print(
    "Result of `df.query('Category == \"Electronics\" and Price > 100')`:\n",
    query_result,
)

Result of `df.query('Category == "Electronics" and Price > 100')`:
    OrderID  Product     Category   Price  Quantity  OrderDate
0      101   Laptop  Electronics  1200.0         1 2025-01-15
3      104  Monitor  Electronics   300.0         2 2025-01-17


In [11]:
# You can use `@` to reach variables from the query
category_of_interest = "Electronics"
query_result_2 = df.query("Category == @category_of_interest")
print(
    f"\nResult of querying with variable '@{category_of_interest}':\n", query_result_2
)


Result of querying with variable '@Electronics':
    OrderID   Product     Category   Price  Quantity  OrderDate
0      101    Laptop  Electronics  1200.0         1 2025-01-15
1      102     Mouse  Electronics    25.5         2 2025-01-15
2      103  Keyboard  Electronics    75.0         1 2025-01-16
3      104   Monitor  Electronics   300.0         2 2025-01-17



---

**Next:** [Handling Missing Data](./07_handling_missing_data.ipynb)